In [1]:
!pip install -q gradio groq tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.3 MB/s eta 0:00:00


In [2]:
import gradio as gr
from groq import Groq
from tavily import TavilyClient
from getpass import getpass

In [3]:
GROQ_API_KEY = getpass("Enter Groq API Key: ")
TAVILY_API_KEY = getpass("Enter Tavily API Key: ")

client = Groq(api_key=GROQ_API_KEY)
tavily = TavilyClient(api_key=TAVILY_API_KEY)

MODEL = "llama-3.1-8b-instant"

Enter Groq API Key: ··········
Enter Tavily API Key: ··········


In [4]:
def diagnose_crop(crop_name, symptoms):

    # Step 1: Disease Prediction

    messages = [
        {
            "role": "system",
            "content": """
You are an expert agricultural scientist.

From the crop symptoms identify:

1. Disease Name
2. One-line reason

Only return these two.
"""
        },
        {
            "role": "user",
            "content": f"""
Crop: {crop_name}

Symptoms:
{symptoms}
"""
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    disease = response.choices[0].message.content

    # Step 2: Tavily Search

    search = tavily.search(
        query=f"{crop_name} {disease} medicine treatment buy",
        max_results=5
    )

    context = ""

    for result in search["results"]:

        context += f"""
Title: {result['title']}
URL: {result['url']}
Content: {result['content']}
"""

    # Step 3: Final Report

    messages = [

        {
            "role":"system",
            "content":"""
You are an agriculture expert.

Generate a beautiful report.

Include:

🌱 Crop Name

🦠 Disease

📝 Reason

💊 Top 3 Medicines

🔗 Medicine Purchase Links

🛡️ Prevention Tips

Keep it neat and beginner friendly.
"""
        },

        {
            "role":"user",
            "content":f"""

Crop:
{crop_name}

Symptoms:
{symptoms}

Disease Prediction:
{disease}

Search Results:
{context}

"""
        }

    ]

    final = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return final.choices[0].message.content

In [5]:
theme = gr.themes.Soft(
    primary_hue="green",
    secondary_hue="emerald",
    neutral_hue="slate"
)

with gr.Blocks(theme=theme,title="🌱 Crop Disease Detector") as demo:

    gr.Markdown(
    """
    # 🌱 AI Crop Disease Detection System

    Diagnose crop diseases using AI 🤖

    Get

    ✅ Disease Name

    ✅ Medicines

    ✅ Purchase Links

    ✅ Prevention Tips
    """
    )

    with gr.Row():

        crop = gr.Textbox(
            label="🌾 Crop Name",
            placeholder="Example: Tomato"
        )

        symptoms = gr.Textbox(
            lines=8,
            label="🍃 Crop Symptoms",
            placeholder="""
Example:

Yellow spots on leaves

Brown circular patches

Leaves curling

Slow plant growth
"""
        )

    btn = gr.Button(
        "🔍 Diagnose",
        variant="primary"
    )

    output = gr.Markdown(label="📄 Report")

    btn.click(
        diagnose_crop,
        inputs=[crop,symptoms],
        outputs=output
    )

/tmp/ipykernel_6366/4197274867.py:7: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme,title="🌱 Crop Disease Detector") as demo:


In [6]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcac81f39ef6198400.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
